# Saudi Real Estate Data Integration

This notebook loads cleaned real estate, rent, sales, and macroeconomic datasets, validates consistency across sources, standardizes region labels, merges demographic and economic features, and exports the final integrated dataset for downstream analysis.

## Setup and Dependencies

Import required libraries and define paths for cleaned and raw datasets. If the environment does not already include the required packages, install them outside this notebook or uncomment the pip command below.

In [ ]:
# Uncomment the following line if pandas or numpy are not installed in the current environment
# !pip install pandas numpy

from pathlib import Path
import pandas as pd
import numpy as np

cleaned_dir = Path('cleaned_dataset')
raw_dir = Path('raw_dataset')

print('Cleaned dataset directory:', cleaned_dir.resolve())
print('Raw dataset directory:', raw_dir.resolve())

## Load Cleaned Datasets

Read the master real estate dataset together with cleaned rent and sales sources. These datasets will be used for validation and integration.

In [ ]:
master = pd.read_csv(cleaned_dir / 'master_real_estate.csv')
rent = pd.read_csv(cleaned_dir / 'rent_cleaned.csv')
sales = pd.read_csv(cleaned_dir / 'sales_cleaned.csv')

print('master:', master.shape)
print('rent:', rent.shape)
print('sales:', sales.shape)

## Explore Master Dataset

Inspect the master dataset structure and quality indicators before enrichment.

In [ ]:
master.info()

print('Shape:', master.shape)
print('Missing values by column:')
print(master.isna().sum())
print('Duplicate rows:', master.duplicated().sum())
print('Unique cities:', sorted(master['city'].dropna().unique()))
print('Property type counts:')
print(master['property_type'].value_counts())

## Aggregate Rent and Sales by Master Grain

Aggregate the cleaned rent and sales datasets to the same time, city, region, and property type granularity as the master dataset. This allows for consistency checks and comparison with the master values.

In [ ]:
aggregate_keys = ['quarter_key', 'year', 'quarter', 'region_ar', 'city', 'property_type']

sales_final = sales.groupby(aggregate_keys, as_index=False).agg({
    'sales_transactions': 'sum',
    'market_value': 'sum',
    'avg_price_m2': 'mean'
})

rent_final = rent.groupby(aggregate_keys, as_index=False).agg({
    'rental_contracts': 'sum',
    'avg_rent': 'mean'
})

print('sales_final:', sales_final.shape)
print('rent_final:', rent_final.shape)
print('Max duplicate groups in sales_final:', sales_final.duplicated(subset=aggregate_keys).sum())
print('Max duplicate groups in rent_final:', rent_final.duplicated(subset=aggregate_keys).sum())

## Load External Economic and Demographic Features

Load inflation, interest rates, population, and average income datasets for enrichment.

In [ ]:
inflation = pd.read_csv(cleaned_dir / 'inflation.csv')
interest_rates = pd.read_csv(cleaned_dir / 'interest_rate.csv')
population = pd.read_csv(cleaned_dir / 'population.csv')
avg_income = pd.read_csv(cleaned_dir / 'average_income.csv')

print('inflation:', inflation.shape)
print('interest_rates:', interest_rates.shape)
print('population:', population.shape)
print('avg_income:', avg_income.shape)

## Normalize Region Labels

Standardize region labels across datasets so merges align correctly. This includes normalizing region names in the master dataset and mapping average income regions to the same region identifiers.

In [ ]:
# Standardize Arabic region names in the master dataset for consistent merges
master["region_ar"] = master["region_ar"].replace({
    "الرياض": "منطقة الرياض",
    "مكة المكرمة": "منطقة مكة المكرمة",
    "تبوك": "منطقة تبوك",
    "جازان": "منطقة جازان",
    "حائل": "منطقة حائل",
    "الجوف": "منطقة الجوف",
    "الباحة": "منطقة الباحة",
    "نجران": "منطقة نجران",
    "القصيم": "منطقة القصيم",
    "الحدود الشماليه": "منطقة الحدود الشماليه",
    "عسير": "منطقة عسير",
    "المدينة المنورة": "منطقة المدينة المنورة",
})

# Convert Arabic region names in the master dataset to English region labels for consistent merging
arabic_to_english = {
    "منطقة الرياض": "Riyadh",
    "منطقة مكة المكرمة": "Makkah",
    "منطقة المدينة المنورة": "Madinah",
    "منطقة القصيم": "Al Qassim",
    "المنطقة الشرقية": "Eastern Province",
    "منطقة عسير": "Aseer",
    "منطقة تبوك": "Tabuk",
    "منطقة حائل": "Hail",
    "منطقة الحدود الشماليه": "Northern Borders",
    "منطقة جازان": "Jazan",
    "منطقة نجران": "Najran",
    "منطقة الباحة": "Al Bahah",
    "منطقة الجوف": "Al Jouf"
}

master = master.copy()
master['region'] = master['region_ar'].replace(arabic_to_english)

# Normalize average income region labels to align with the master dataset's English region names
income_region_map = {
    'Qassim': 'Al Qassim',
    'Asir': 'Aseer',
    'AL - Baha': 'Al Bahah',
    'AL - Jouf': 'Al Jouf'
}
avg_income['Region'] = avg_income['Region'].replace(income_region_map)
avg_income = avg_income.rename(columns={'Region': 'region', 'Average_Income': 'avg_income'})

# Standardize interest rate quarter keys to match the master dataset format
interest_rates['quarter_key'] = interest_rates['quarter_key'].astype(str).str.replace('QQ', 'Q', regex=False)

print('Normalized master regions:', sorted(master['region'].dropna().unique()))
print('Sample average income regions:', sorted(avg_income['region'].dropna().unique()))
print('Sample interest rate quarter keys:', sorted(interest_rates['quarter_key'].unique())[:10])

## Merge Enrichment Features Into Master Dataset

Merge population, average income, inflation, and repo rate data into the master dataset using the appropriate keys.

In [ ]:
master = master.merge(population[['city', 'population']], on='city', how='left')
master = master.merge(avg_income[['region', 'avg_income']], on='region', how='left')
master = master.merge(inflation, on='year', how='left')
master = master.merge(interest_rates[['quarter_key', 'repo_rate']], on='quarter_key', how='left')

print('Merged master shape:', master.shape)
print('Columns added:', [col for col in ['population', 'avg_income', 'inflation_rate', 'repo_rate'] if col in master.columns])

## Validate Final Dataset

Check for missing values in the newly merged columns and confirm the final dataset structure is ready for export.

In [ ]:
validation_columns = ['population', 'avg_income', 'inflation_rate', 'repo_rate']
validation_results = {col: int(master[col].isna().sum()) for col in validation_columns if col in master.columns}
print('Missing values after merge:')
print(validation_results)

master.head()

## Export Final Integrated Dataset

Save the fully enriched dataset to the cleaned dataset directory for downstream analytics and modeling.

In [ ]:
master.to_csv(cleaned_dir / 'master_real_estate_market.csv', index=False, encoding='utf-8-sig')
print('Export complete:', cleaned_dir / 'master_real_estate_market.csv')